# Deep Generative Models: Diffusion, GANs, VAEs, and Energy-Based Models
### Advanced Machine Learning and Data Analysis for the Physical Sciences
**Morten Hjorth-Jensen**, University of Oslo / Michigan State University

Date: **May 2026**

---

This notebook gives a unified treatment of the four principal families of
deep generative models, implemented in PyTorch on MNIST:

| Section | Model | Key idea |
|---------|-------|----------|
| **1** | **Variational Autoencoders (VAE)** | Variational ELBO, reparameterisation trick |
| **2** | **Denoising Diffusion Probabilistic Models (DDPM)** | Hierarchical ELBO, noise prediction |
| **3** | **Generative Adversarial Networks (GAN)** | Minimax game, Jensen-Shannon divergence |
| **4** | **Comparison: Energy-Based, VAE, Diffusion, GAN** | Pros, cons, when to use each |
| **5** | **Exercises** | Mathematical and computational |

> All models are implemented from scratch — no external generative-model
> library is required. Code maps directly to the mathematics.

---

### References

- Kingma & Welling, *Auto-Encoding Variational Bayes*, arXiv:1312.6114 (2014)
- Ho, Jain & Abbeel, *Denoising Diffusion Probabilistic Models*, arXiv:2006.11239 (2020)
- Goodfellow et al., *Generative Adversarial Nets*, NeurIPS 2014
- Song & Ermon, *Score-Based Generative Modeling*, NeurIPS 2020
- Luo, *Understanding Diffusion Models: A Unified Perspective*, arXiv:2208.11970 (2022)
- Hinton et al., *A Fast Learning Algorithm for Deep Belief Nets*, Neural Computation 2006


---
## Section 1 — Variational Autoencoders

### 1.1  Motivation: the intractability problem

Our goal is to train a generative model $p_{\boldsymbol{\theta}}(\boldsymbol{x})$
by maximising $\log p_{\boldsymbol{\theta}}(\boldsymbol{x})$ over the training data.
The marginal over a latent variable $\boldsymbol{h}$,

$$p(\boldsymbol{x}) = \int p_{\boldsymbol{\theta}}(\boldsymbol{x}|\boldsymbol{h})\,p(\boldsymbol{h})\,d\boldsymbol{h},$$

is intractable for deep networks.

### 1.2  The Evidence Lower BOund (ELBO)

Variational inference introduces an approximate posterior
$q_{\boldsymbol{\phi}}(\boldsymbol{h}|\boldsymbol{x})$ and derives:

$$
\log p(\boldsymbol{x})
\geq \underbrace{\mathbb{E}_{q_{\boldsymbol{\phi}}}\![\log p_{\boldsymbol{\theta}}(\boldsymbol{x}|\boldsymbol{h})]}_{{\text{reconstruction}}}
- \underbrace{D_{\mathrm{KL}}\!\left(q_{\boldsymbol{\phi}}(\boldsymbol{h}|\boldsymbol{x})\,\|\,p(\boldsymbol{h})\right)}_{{\text{regularisation}}}
=: \mathcal{L}(\boldsymbol{\phi},\boldsymbol{\theta}).
$$

The bound is tight when $q_{\boldsymbol{\phi}} = p(\boldsymbol{h}|\boldsymbol{x})$.

### 1.3  Reparameterisation trick

With Gaussian encoder
$q_{\boldsymbol{\phi}}(\boldsymbol{h}|\boldsymbol{x})=\mathcal{N}(\boldsymbol{\mu}_{\boldsymbol{\phi}},\boldsymbol{\sigma}^2_{\boldsymbol{\phi}}\mathbf{I})$
and prior $p(\boldsymbol{h})=\mathcal{N}(\mathbf{0},\mathbf{I})$:

$$\boldsymbol{h} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\boldsymbol{x})
  + \boldsymbol{\sigma}_{\boldsymbol{\phi}}(\boldsymbol{x})\odot\boldsymbol{\epsilon},
  \qquad \boldsymbol{\epsilon}\sim\mathcal{N}(\mathbf{0},\mathbf{I}).$$

The closed-form KL divergence is:

$$D_{\mathrm{KL}}(q_{\boldsymbol{\phi}}\,\|\,p)
= \tfrac{1}{2}\sum_{j=1}^{d_h}\!\left(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1\right).$$

### 1.4  Imports and setup


In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────
import math, time, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

# ── Shared data loading ────────────────────────────────────────────────────
# VAE uses [0,1] (Bernoulli decoder).  DDPM uses [-1,1] (Gaussian).
transform_01 = transforms.Compose([
    transforms.ToTensor(),          # -> [0, 1]
])
transform_11 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # -> [-1, 1]
])

BATCH = 128


train_01 = datasets.MNIST('.', train=True,  download=True, transform=transform_01)
test_01  = datasets.MNIST('.', train=False, download=True, transform=transform_01)
train_11 = datasets.MNIST('.', train=True,  download=True, transform=transform_11)
test_11  = datasets.MNIST('.', train=False, download=True, transform=transform_11)

loader_train_01 = DataLoader(train_01, batch_size=BATCH, shuffle=True,  num_workers=0)
loader_test_01  = DataLoader(test_01,  batch_size=BATCH, shuffle=False, num_workers=0)
loader_train_11 = DataLoader(train_11, batch_size=BATCH, shuffle=True,  num_workers=0)

print(f'Train: {len(train_01)} | Test: {len(test_01)}')
print(f'Image shape: {train_01[0][0].shape}')


### 1.5  VAE architecture

In [ ]:
# ── VAE Model ─────────────────────────────────────────────────────────────
class VAE(nn.Module):
    """
    Convolutional VAE for 28x28 grayscale images (MNIST).

    Encoder:  x (1x28x28) -> Conv stack -> mu, log_var  (LATENT_DIM,)
    Decoder:  z (LATENT_DIM,) -> ConvTranspose stack -> x_hat (1x28x28)

    The decoder outputs logits for Bernoulli(sigmoid(logit)), so we use
    binary cross-entropy reconstruction loss.
    """
    def __init__(self, latent_dim: int = 20):
        super().__init__()
        self.latent_dim = latent_dim

        # ── Encoder ────────────────────────────────────────────────────────
        # Input: (B, 1, 28, 28)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1),   # -> (B,32,14,14)
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # -> (B,64, 7, 7)
            nn.LeakyReLU(0.2),
            nn.Flatten(),                                # -> (B, 64*7*7)
        )
        enc_flat = 64 * 7 * 7
        self.fc_mu      = nn.Linear(enc_flat, latent_dim)
        self.fc_log_var = nn.Linear(enc_flat, latent_dim)

        # ── Decoder ────────────────────────────────────────────────────────
        self.fc_dec = nn.Linear(latent_dim, enc_flat)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # -> (B,32,14,14)
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,  1, 4, stride=2, padding=1),  # -> (B,1,28,28)
            # No final activation — we output logits for BCEWithLogitsLoss
        )

    # -- Inference network ------------------------------------------------
    def encode(self, x):
        """x: (B,1,28,28) -> mu, log_var: (B, latent_dim)"""
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_log_var(h)

    # -- Reparameterisation trick -----------------------------------------
    def reparameterise(self, mu, log_var):
        """
        h = mu + sigma * eps,  eps ~ N(0,I)
        std = exp(0.5 * log_var)  ->  var = sigma^2
        """
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    # -- Generative network -----------------------------------------------
    def decode(self, z):
        """z: (B, latent_dim) -> logits: (B,1,28,28)"""
        h = self.fc_dec(z)
        return self.decoder(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z           = self.reparameterise(mu, log_var)
        x_hat       = self.decode(z)
        return x_hat, mu, log_var


# ── ELBO loss ──────────────────────────────────────────────────────────────
def vae_loss(x, x_hat_logits, mu, log_var, beta: float = 1.0):
    """
    ELBO = -Reconstruction + beta * KL

    Reconstruction:  BCE(sigmoid(x_hat_logits), x)   [pixels in [0,1]]
    KL:              0.5 * sum(mu^2 + sigma^2 - log_sigma^2 - 1)

    beta controls the KL weight (beta-VAE; beta=1 is the standard VAE).
    """
    B = x.size(0)
    # Flatten for BCE
    recon = F.binary_cross_entropy_with_logits(
        x_hat_logits.view(B, -1), x.view(B, -1), reduction='sum') / B
    # Closed-form KL: -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    kl = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum() / B
    return recon + beta * kl, recon, kl


# Instantiate
vae = VAE(latent_dim=20).to(device)
n_params_vae = sum(p.numel() for p in vae.parameters() if p.requires_grad)
print(f'VAE parameters: {n_params_vae:,}')
print(vae)


### 1.6  VAE training

In [ ]:
def train_vae(model, loader, epochs=20, lr=1e-3, beta=1.0):
    """
    Train VAE for `epochs` epochs.
    Returns history dict: {'elbo', 'recon', 'kl'} lists of per-epoch means.
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'elbo': [], 'recon': [], 'kl': []}
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        tot_elbo = tot_recon = tot_kl = 0.0
        for x, _ in loader:
            x = x.to(device)
            x_hat, mu, lv = model(x)
            loss, recon, kl = vae_loss(x, x_hat, mu, lv, beta=beta)
            opt.zero_grad(); loss.backward(); opt.step()
            tot_elbo  += loss.item()
            tot_recon += recon.item()
            tot_kl    += kl.item()

        n = len(loader)
        history['elbo'].append(tot_elbo / n)
        history['recon'].append(tot_recon / n)
        history['kl'].append(tot_kl / n)

        if ep % 5 == 0 or ep == 1:
            print(f'  Epoch {ep:3d}/{epochs}  ELBO={history["elbo"][-1]:.2f}  '
                  f'Recon={history["recon"][-1]:.2f}  KL={history["kl"][-1]:.2f}')

    elapsed = time.time() - t0
    print(f'Training time: {elapsed:.1f}s')
    history['wall_s'] = elapsed
    return history


print('Training VAE on MNIST (20 epochs)...')
print('(Uses [0,1]-normalised pixels and Bernoulli decoder)')
vae_history = train_vae(vae, loader_train_01, epochs=20, lr=1e-3, beta=1.0)


### 1.7  Reconstructions and random samples

In [ ]:
def show_images(tensor, title='', nrow=8):
    """Display a batch of images from a tensor in [-1,1] or [0,1]."""
    tensor = tensor.detach().cpu().clamp(0, 1)
    grid   = make_grid(tensor, nrow=nrow, normalize=False, pad_value=0.5)
    fig, ax = plt.subplots(figsize=(12, 2.5))
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    ax.set_title(title, fontsize=13)
    ax.axis('off'); plt.tight_layout(); plt.show()


# ── Reconstructions ────────────────────────────────────────────────────────
vae.eval()
with torch.no_grad():
    x_real, _ = next(iter(loader_test_01))
    x_real = x_real[:16].to(device)
    x_hat_logits, mu, lv = vae(x_real)
    x_hat = torch.sigmoid(x_hat_logits)   # logits -> probs in [0,1]

show_images(x_real, 'VAE — Original test images')
show_images(x_hat,  'VAE — Reconstructions')

# ── Random samples ─────────────────────────────────────────────────────────
with torch.no_grad():
    z = torch.randn(16, vae.latent_dim, device=device)
    samples = torch.sigmoid(vae.decode(z))

show_images(samples, 'VAE — Random samples (z ~ N(0,I))')


### 1.8  Latent space visualisation (2-D projection)

In [ ]:
# ── Encode all test images and plot first two latent dimensions ────────────
vae.eval()
all_mu, all_labels = [], []
with torch.no_grad():
    for x, y in loader_test_01:
        mu, _ = vae.encode(x.to(device))
        all_mu.append(mu.cpu())
        all_labels.append(y)

all_mu     = torch.cat(all_mu).numpy()
all_labels = torch.cat(all_labels).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2-D scatter of first two latent dims
sc = axes[0].scatter(all_mu[:, 0], all_mu[:, 1],
                     c=all_labels, cmap='tab10', s=3, alpha=0.5)
plt.colorbar(sc, ax=axes[0])
axes[0].set_title('VAE latent space ($\\mu_1$ vs $\\mu_2$)')
axes[0].set_xlabel('$z_1$'); axes[0].set_ylabel('$z_2$')

# Training curves
epochs_range = range(1, len(vae_history['elbo']) + 1)
axes[1].plot(epochs_range, vae_history['elbo'],  label='ELBO', lw=2)
axes[1].plot(epochs_range, vae_history['recon'], label='Reconstruction', lw=2, ls='--')
axes[1].plot(epochs_range, vae_history['kl'],    label='KL', lw=2, ls=':')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('VAE Training Curves')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


### 1.9  Latent interpolation

In [ ]:
# ── Interpolate between two test-set digits ────────────────────────────────
vae.eval()
with torch.no_grad():
    # Pick a '3' and a '7' from the test set
    xs, ys = [], []
    for x, y in test_01:
        if y == 3 and len([v for v in ys if v==3]) < 1:
            xs.append(x); ys.append(y)
        if y == 7 and len([v for v in ys if v==7]) < 1:
            xs.append(x); ys.append(y)
        if len(xs) == 2:
            break

    x_a = xs[0].unsqueeze(0).to(device)
    x_b = xs[1].unsqueeze(0).to(device)
    mu_a, _ = vae.encode(x_a)
    mu_b, _ = vae.encode(x_b)

    # Linear interpolation in latent space
    steps = 10
    interp = [torch.sigmoid(vae.decode((1-t)*mu_a + t*mu_b))
              for t in torch.linspace(0, 1, steps)]
    interp_grid = torch.cat([x_a.cpu()] +
                            [img.cpu() for img in interp] +
                            [x_b.cpu()], dim=0)

show_images(interp_grid,
    f'VAE latent interpolation: digit {ys[0]} (left) -> digit {ys[1]} (right)',
    nrow=steps+2)


---
## Section 2 — Denoising Diffusion Probabilistic Models (DDPM)

### 2.1  Mathematical foundations

The DDPM (Ho et al. 2020) is a **hierarchical VAE with $T$ latent levels
and a fixed (non-learned) encoder**.

#### Forward process

A fixed Markov chain that gradually destroys structure by adding Gaussian noise:

$$q(\boldsymbol{x}_t|\boldsymbol{x}_{t-1})
= \mathcal{N}\!\left(\boldsymbol{x}_t;\,\sqrt{\alpha_t}\,\boldsymbol{x}_{t-1},\,(1-\alpha_t)\mathbf{I}\right),$$

where $\alpha_t = 1-\beta_t$ and $\beta_t$ is the noise schedule
(small positive values, typically from $10^{-4}$ to $0.02$).

#### Closed-form marginal (key result)

By induction on the Gaussian product rule,
$q(\boldsymbol{x}_t|\boldsymbol{x}_{t-1})$ composed $t$ times gives:

$$\boxed{q(\boldsymbol{x}_t|\boldsymbol{x}_0)
= \mathcal{N}\!\left(\boldsymbol{x}_t;\,\sqrt{\bar\alpha_t}\,\boldsymbol{x}_0,\,
(1-\bar\alpha_t)\mathbf{I}\right)},
\qquad \bar\alpha_t = \prod_{s=1}^t\alpha_s.$$

This allows sampling $\boldsymbol{x}_t$ in a **single step**:
$\boldsymbol{x}_t = \sqrt{\bar\alpha_t}\,\boldsymbol{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol{\epsilon}$,
$\boldsymbol{\epsilon}\sim\mathcal{N}(\mathbf{0},\mathbf{I})$.

#### Training objective (noise prediction)

The full ELBO decomposes into $T$ terms.
After re-weighting (Ho et al. Appendix A), the simplified objective is:

$$\mathcal{L}_{\text{simple}}
= \mathbb{E}_{t,\boldsymbol{x}_0,\boldsymbol{\epsilon}}\!\left[
  \|\boldsymbol{\epsilon}
    - \boldsymbol{\epsilon}_{\boldsymbol{\theta}}\!\left(\sqrt{\bar\alpha_t}\boldsymbol{x}_0
      + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon},\,t\right)\|^2\right].$$

The network $\boldsymbol{\epsilon}_{\boldsymbol{\theta}}$ predicts the noise
$\boldsymbol{\epsilon}$ added at timestep $t$, not the image directly.

#### Reverse step (ancestral sampling)

$$\boldsymbol{x}_{t-1} =
\frac{1}{\sqrt{\alpha_t}}\!\left(\boldsymbol{x}_t
- \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\boldsymbol{\epsilon}_{\boldsymbol{\theta}}(\boldsymbol{x}_t,t)\right)
+ \sqrt{\tilde\beta_t}\,\boldsymbol{z},
\quad \boldsymbol{z}\sim\mathcal{N}(\mathbf{0},\mathbf{I}),$$

where $\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$
is the posterior variance.

#### Connection to score matching

The noise-prediction objective is equivalent to denoising score matching:

$$\boldsymbol{\epsilon}_{\boldsymbol{\theta}}(\boldsymbol{x}_t,t)
= -\sqrt{1-\bar\alpha_t}\,\nabla_{\boldsymbol{x}_t}\log q(\boldsymbol{x}_t).$$

Thus DDPM implicitly learns the **score function** (gradient of log density)
at each noise level — connecting it to score-based generative models
(Song & Ermon, 2020).


### 2.2  Noise schedule and forward process

In [ ]:
# ── Diffusion hyperparameters ──────────────────────────────────────────────
T          = 300          # number of diffusion timesteps
BETA_START = 1e-4
BETA_END   = 0.02
IMG_SIZE   = 28
CHANNELS   = 1

# ── Linear noise schedule ──────────────────────────────────────────────────
betas          = torch.linspace(BETA_START, BETA_END, T).to(device)
alphas         = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)   # alpha_bar_t
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

# Quantities used in sampling
sqrt_acp        = alphas_cumprod.sqrt()
sqrt_1m_acp     = (1.0 - alphas_cumprod).sqrt()
sqrt_recip_a    = (1.0 / alphas).sqrt()
# Posterior variance: beta_tilde = (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t) * beta_t
posterior_var   = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

print('Noise schedule summary:')
print(f'  T={T}, beta in [{BETA_START:.1e}, {BETA_END:.2f}]')
print(f'  sqrt(alpha_bar) at t=0   : {float(sqrt_acp[0]):.4f}  (signal weight)')
print(f'  sqrt(alpha_bar) at t=T/2 : {float(sqrt_acp[T//2]):.4f}')
print(f'  sqrt(alpha_bar) at t=T-1 : {float(sqrt_acp[-1]):.4f}  (nearly 0 = pure noise)')

# ── Visualise the forward noising process ──────────────────────────────────
def q_sample(x0, t, noise=None):
    """
    Sample x_t from q(x_t | x_0) using the closed-form marginal.
    x0: (B,C,H,W),  t: (B,) integer timesteps
    """
    if noise is None:
        noise = torch.randn_like(x0)
    t_idx   = t.long() if hasattr(t, "long") else t
    s_acp   = sqrt_acp[t_idx].view(-1, 1, 1, 1)
    s_1m    = sqrt_1m_acp[t_idx].view(-1, 1, 1, 1)
    return s_acp * x0 + s_1m * noise, noise


x0_demo, _ = next(iter(loader_train_11))
x0_demo    = x0_demo[:1].to(device)   # single image

timesteps_show = [0, 49, 99, 149, 199, 249, 299]
noised_imgs = []
for t_val in timesteps_show:
    t_tensor = torch.tensor([t_val], device=device)
    xt, _    = q_sample(x0_demo, t_tensor)
    noised_imgs.append(xt.squeeze().cpu())

fig, axes = plt.subplots(1, len(timesteps_show), figsize=(14, 2.5))
for ax, img, t_val in zip(axes, noised_imgs, timesteps_show):
    ax.imshow(img.clamp(-1,1)*0.5+0.5, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f't={t_val}', fontsize=11)
    ax.axis('off')
fig.suptitle('Forward diffusion: $q(\\mathbf{x}_t|\\mathbf{x}_0)$',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 2.3  Sinusoidal time embedding

In [ ]:
class SinusoidalPosEmb(nn.Module):
    """
    Maps a scalar timestep t (normalised to [0,1]) to a d-dimensional
    sinusoidal embedding — analogous to transformer positional encoding.

    emb[2i]   = sin(t * omega_i)
    emb[2i+1] = cos(t * omega_i)
    where omega_i = exp(-log(10000) * i / (half - 1)).
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t):          # t: (B,)  float in [0, 1]
        half  = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) *
            torch.arange(half, device=t.device, dtype=torch.float32)
            / max(half - 1, 1)
        )                          # (half,)
        args  = t.unsqueeze(1) * freqs.unsqueeze(0)   # (B, half)
        return torch.cat([args.sin(), args.cos()], dim=-1)   # (B, dim)


# Quick sanity check
emb_test = SinusoidalPosEmb(128)(torch.linspace(0, 1, 10))
print(f'Sinusoidal embedding: input (10,) -> output {emb_test.shape}')


### 2.4  U-Net denoiser architecture

In [ ]:
class ResBlock(nn.Module):
    """
    Residual conv block with GroupNorm and time-embedding injection.
    Implements: x -> Conv -> GN -> SiLU -> Conv -> GN + time_proj + skip
    """
    def __init__(self, in_ch, out_ch, emb_dim):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch), nn.SiLU()
        )
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(emb_dim, out_ch))
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch), nn.SiLU()
        )
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.res_conv(x)


class UNet(nn.Module):
    """
    Lightweight U-Net for noise prediction on 28x28 images.

    Architecture:
        Encoder: 3 ResBlocks with downsampling
        Bottleneck: 1 ResBlock
        Decoder: 3 ResBlocks with upsampling + skip connections
        Output conv: -> (C, H, W) predicted noise

    Time conditioning: sinusoidal embedding -> MLP -> injected in every ResBlock.
    """
    def __init__(self, in_ch=1, base_ch=64, emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(emb_dim),
            nn.Linear(emb_dim, emb_dim * 2),
            nn.SiLU(),
            nn.Linear(emb_dim * 2, emb_dim),
        )
        ch = base_ch
        # ── Encoder ────────────────────────────────────────────────────────
        self.enc1 = ResBlock(in_ch, ch,      emb_dim)   # 28x28
        self.down1 = nn.Conv2d(ch, ch,  3, stride=2, padding=1)  # 14x14
        self.enc2 = ResBlock(ch,  ch*2,  emb_dim)   # 14x14
        self.down2 = nn.Conv2d(ch*2, ch*2, 3, stride=2, padding=1) # 7x7
        self.enc3 = ResBlock(ch*2, ch*4, emb_dim)   # 7x7
        # ── Bottleneck ─────────────────────────────────────────────────────
        self.mid  = ResBlock(ch*4, ch*4, emb_dim)   # 7x7
        # ── Decoder ────────────────────────────────────────────────────────
        self.up2  = nn.ConvTranspose2d(ch*4, ch*2, 4, stride=2, padding=1)  # 14x14
        self.dec2 = ResBlock(ch*4, ch*2, emb_dim)   # 14x14 (with skip)
        self.up1  = nn.ConvTranspose2d(ch*2, ch,   4, stride=2, padding=1)  # 28x28
        self.dec1 = ResBlock(ch*2, ch,   emb_dim)   # 28x28 (with skip)
        self.out  = nn.Conv2d(ch, in_ch, 1)

    def forward(self, x, t):
        """x: (B,C,H,W),  t: (B,) float in [0,1]"""
        te = self.time_mlp(t)          # (B, emb_dim)
        # Encoder
        e1 = self.enc1(x,  te)                        # (B, ch, 28, 28)
        e2 = self.enc2(self.down1(e1), te)             # (B,2ch, 14, 14)
        e3 = self.enc3(self.down2(e2), te)             # (B,4ch,  7,  7)
        # Bottleneck
        m  = self.mid(e3, te)                          # (B,4ch,  7,  7)
        # Decoder with skip connections
        d2 = self.dec2(torch.cat([self.up2(m), e2], dim=1), te)  # (B,2ch,14,14)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1), te) # (B, ch,28,28)
        return self.out(d1)                            # (B,C,28,28)


ddpm_model   = UNet(in_ch=CHANNELS, base_ch=64, emb_dim=128).to(device)
n_params_ddpm = sum(p.numel() for p in ddpm_model.parameters() if p.requires_grad)
print(f'DDPM U-Net parameters: {n_params_ddpm:,}')


### 2.5  DDPM training

In [ ]:
def diffusion_loss(model, x0):
    """
    Sample t ~ Uniform[0,T-1], eps ~ N(0,I),
    corrupt x0 -> x_t using the closed-form forward marginal,
    predict eps with the model, return MSE.

    This directly optimises L_simple from Ho et al. (2020).
    """
    B     = x0.size(0)
    t     = torch.randint(0, T, (B,), device=device)
    x_t, noise = q_sample(x0, t)
    t_in  = t.float() / T             # normalise to [0, 1]
    pred  = model(x_t, t_in)
    return F.mse_loss(pred, noise)


def train_ddpm(model, loader, epochs=20, lr=2e-4):
    """Train DDPM; return history dict."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': []}
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        for x, _ in loader:
            loss = diffusion_loss(model, x.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()
        history['loss'].append(tot / len(loader))
        if ep % 5 == 0 or ep == 1:
            print(f'  Epoch {ep:3d}/{epochs}  MSE={history["loss"][-1]:.5f}')

    history['wall_s'] = time.time() - t0
    print(f'Training time: {history["wall_s"]:.1f}s')
    return history


print('Training DDPM on MNIST (20 epochs)...')
print('(Uses [-1,1]-normalised pixels; predicts noise)')
ddpm_history = train_ddpm(ddpm_model, loader_train_11, epochs=20, lr=2e-4)


### 2.6  DDPM sampling

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples=16, verbose=True):
    """
    Full DDPM reverse process: x_T -> x_0.
    Each step:
        mu = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-alpha_bar_t) * eps_hat)
        x_{t-1} = mu + sqrt(beta_tilde_t) * z  (z=0 at t=1)
    """
    model.eval()
    shape = (n_samples, CHANNELS, IMG_SIZE, IMG_SIZE)
    x     = torch.randn(*shape).to(device)

    for i in reversed(range(T)):
        t_tensor  = torch.full((n_samples,), i, device=device).float() / T
        eps_hat   = model(x, t_tensor)

        # Wrap scalar-indexed precomputed values as 0-dim tensors
        beta_t    = torch.tensor(betas[i])
        s_recip_a = torch.tensor(sqrt_recip_a[i])
        s_1m_acp  = torch.tensor(sqrt_1m_acp[i])
        pvar      = torch.tensor(posterior_var[i])

        # Mean of p_theta(x_{t-1} | x_t)
        mu = s_recip_a * (x - beta_t / s_1m_acp * eps_hat)
        if i > 0:
            mu = mu + pvar.sqrt() * torch.randn_like(x)
        x = mu

        if verbose and i % 50 == 0:
            print(f'  t={i:4d}  |x| range: [{x.min():.2f}, {x.max():.2f}]')

    return x.clamp(-1, 1)


print('Generating 16 DDPM samples (T=300 reverse steps)...')
ddpm_samples = ddpm_sample(ddpm_model, n_samples=16, verbose=True)

# Convert from [-1,1] to [0,1] for display
ddpm_samples_01 = (ddpm_samples * 0.5 + 0.5).cpu()
show_images(ddpm_samples_01, 'DDPM — Random samples after reverse diffusion')


### 2.7  Visualising the reverse diffusion trajectory

In [ ]:
@torch.no_grad()
def ddpm_sample_trajectory(model, n_img=4):
    """
    Sample with snapshots saved at regular intervals.
    Returns dict of {timestep: images}.
    """
    model.eval()
    shape = (n_img, CHANNELS, IMG_SIZE, IMG_SIZE)
    x     = torch.randn(*shape).to(device)
    traj  = {}
    snap_at = set([T-1, T//4*3, T//2, T//4, 10, 0])

    for i in reversed(range(T)):
        t_in    = torch.full((n_img,), i, device=device).float() / T
        eps_hat = model(x, t_in)
        beta_t  = torch.tensor(betas[i])
        mu      = torch.tensor(sqrt_recip_a[i]) * (
                      x - beta_t / torch.tensor(sqrt_1m_acp[i]) * eps_hat)
        if i > 0:
            mu = mu + torch.tensor(posterior_var[i]).sqrt() * torch.randn_like(x)
        x = mu
        if i in snap_at:
            traj[i] = (x.clamp(-1, 1) * 0.5 + 0.5).cpu()

    return traj


print('Recording reverse diffusion trajectory...')
traj = ddpm_sample_trajectory(ddpm_model, n_img=4)

snap_keys = sorted(traj.keys(), reverse=True)
fig, axes = plt.subplots(4, len(snap_keys), figsize=(14, 7))
for col, t_key in enumerate(snap_keys):
    imgs = traj[t_key]
    for row in range(4):
        axes[row, col].imshow(imgs[row].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[0, col].set_title(f't={t_key}', fontsize=10)

fig.suptitle('DDPM reverse diffusion: $x_T \\to x_0$ (rows = 4 independent samples)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


### 2.8  DDIM faster sampling

In [ ]:
@torch.no_grad()
def ddim_sample(model, n_samples=16, n_steps=50, eta=0.0):
    """
    DDIM (Song et al. 2020) deterministic sampler.

    Instead of stochastic DDPM reverse steps, solves the probability-flow ODE:
        x_{t-1} = sqrt(alpha_bar_{t-1}) * x0_hat
                + sqrt(1 - alpha_bar_{t-1} - sigma_t^2) * eps_hat
                + sigma_t * z

    With eta=0 (fully deterministic), sigma_t=0.
    Uses a subsequence of T timesteps of length n_steps.
    """
    model.eval()
    shape = (n_samples, CHANNELS, IMG_SIZE, IMG_SIZE)
    x     = torch.randn(*shape).to(device)

    # Choose evenly-spaced timestep subsequence
    step_indices = torch.linspace(T - 1, 0, n_steps).long()

    for i in range(len(step_indices) - 1):
        t_curr = step_indices[i].item()
        t_prev = step_indices[i + 1].item()

        t_in    = torch.full((n_samples,), t_curr, device=device).float() / T
        eps_hat = model(x, t_in)

        acp_t    = torch.tensor(alphas_cumprod[t_curr])
        acp_prev = torch.tensor(alphas_cumprod[t_prev])

        # Predict x_0 from x_t and eps
        x0_hat = (x - (1 - acp_t).sqrt() * eps_hat) / acp_t.sqrt()
        x0_hat = x0_hat.clamp(-1, 1)

        # Direction pointing to x_t
        dir_xt = (1 - acp_prev).sqrt() * eps_hat

        # Noise (zero when eta=0 = deterministic)
        sigma_t = eta * ((1 - acp_prev) / (1 - acp_t) * (1 - acp_t/acp_prev)).sqrt()
        noise   = torch.randn_like(x) if (eta > 0 and t_prev > 0) else torch.zeros_like(x)

        x = acp_prev.sqrt() * x0_hat + dir_xt + sigma_t * noise

    return x.clamp(-1, 1)


import time as _time

# Compare DDPM (T=300 steps) vs DDIM (50 steps)
t0 = _time.time()
ddpm_s = ddpm_sample(ddpm_model, n_samples=8, verbose=False)
t_ddpm = _time.time() - t0

t0 = _time.time()
ddim_s = ddim_sample(ddpm_model, n_samples=8, n_steps=50, eta=0.0)
t_ddim = _time.time() - t0

print(f'DDPM ({T} steps): {t_ddpm:.2f}s  |  DDIM (50 steps): {t_ddim:.2f}s  '
      f'({T/50:.0f}x speedup)')

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for col in range(8):
    for row, (title, imgs) in enumerate([
        (f'DDPM T={T}',       (ddpm_s*0.5+0.5).cpu()),
        ('DDIM n=50 (eta=0)', (ddim_s*0.5+0.5).cpu()),
    ]):
        axes[row, col].imshow(imgs[col].squeeze().clamp(0,1), cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, 0].set_ylabel(title, fontsize=10)

plt.suptitle(f'DDPM ({T} steps) vs DDIM (50 steps) — same trained model',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## Section 3 — Generative Adversarial Networks (GAN)

### 3.1  The adversarial framework

Unlike VAEs and diffusion models, GANs do **not** optimise a likelihood bound.
Instead, two networks play a **zero-sum game**:

| Network | Input | Output | Goal |
|---------|-------|--------|------|
| **Generator** $G$ | Noise $\mathbf{z}\sim p_z$ | Synthetic sample $G(\mathbf{z})$ | Fool $D$ |
| **Discriminator** $D$ | Image $\mathbf{x}$ | $D(\mathbf{x})\in(0,1)$ | Detect fakes |

### 3.2  Minimax objective

$$\boxed{\min_G\,\max_D\;V(G,D)
= \underbrace{\mathbb{E}_{\mathbf{x}\sim p_r}[\log D(\mathbf{x})]}_{\text{$D$ correct on real}}
+ \underbrace{\mathbb{E}_{\mathbf{z}\sim p_z}[\log(1-D(G(\mathbf{z})))]}_{\text{$D$ correct on fake}}}$$

### 3.3  Optimal discriminator

For fixed $G$, the discriminator's integrand $f(t)=A\log t+B\log(1-t)$ is
maximised pointwise at:

$$D^*(\mathbf{x}) = \frac{p_r(\mathbf{x})}{p_r(\mathbf{x})+p_g(\mathbf{x})}.$$

This is the Bayes-optimal classifier. At the Nash equilibrium $p_g=p_r$:
$D^*\equiv 1/2$.

### 3.4  Connection to Jensen-Shannon divergence

Substituting $D^*$ into $V$ and expanding the JS divergence:

$$\boxed{V(G,D^*) = 2\,D_{\mathrm{JS}}(p_r\|p_g) - 2\log 2.}$$

The GAN minimax objective is equivalent to minimising the Jensen-Shannon
divergence between $p_r$ and $p_g$.
The minimum $-2\log 2$ is achieved when $p_g=p_r$.

### 3.5  Non-saturating generator loss

The original loss $\log(1-D(G(\mathbf{z})))$ vanishes when $D$ is strong.
The non-saturating alternative:

$$\mathcal{L}_G^{\text{non-sat}} = -\mathbb{E}[\log D(G(\mathbf{z}))],
\qquad |\text{gradient}|\propto 1/D(G(\mathbf{z}))\to\infty\text{ as }D(G(\mathbf{z}))\to 0.$$

### 3.6  Imports, hyperparameters, and device setup


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device    : {device}")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


### 3.7  Hyperparameters

In [ ]:
# Architecture
IMAGE_SIZE   = (28, 28)
IMAGE_DIM    = 28 * 28       # = 784
Z_DIM        = 100           # latent noise dimension
MODE_Z       = 'normal'      # 'normal' or 'uniform'
G_HIDDEN     = 256           # generator hidden units (per block)
D_HIDDEN     = 256           # discriminator hidden units (per block)

# Training
BATCH_SIZE   = 128
NUM_EPOCHS   = 50
LR_G         = 2e-4
LR_D         = 2e-4
BETA1        = 0.5           # Adam beta_1 -- lower than default to reduce momentum
LABEL_SMOOTH = 0.9           # one-sided label smoothing for real targets


### 3.8  Generator and Discriminator architectures

**Generator**: $\mathbf{z}\in\mathbb{R}^{100}
\xrightarrow{\text{Linear+BN+LReLU}\times 3}
\mathbb{R}^{784}$  (Tanh output, matched to $[-1,1]$ MNIST).

**Discriminator**: $\mathbb{R}^{784}
\xrightarrow{\text{Linear+LReLU+Dropout}\times 3}
(0,1)$  (Sigmoid output).

BatchNorm stabilises $G$; absent in $D$ to avoid minibatch dependencies.
LeakyReLU(0.2) prevents dying ReLU. Weights from $\mathcal{N}(0,0.02^2)$ (DCGAN).


In [ ]:
class Generator(nn.Module):
    """
    Fully-connected GAN generator.

    z in R^{Z_DIM} -> (Linear + BN + LeakyReLU) x 3 -> Linear -> Tanh
    Output: flattened 28x28 image in [-1, 1].

    BatchNorm stabilises training; Tanh matches the [-1,1] MNIST normalisation.
    """
    def __init__(self, z_dim=Z_DIM, hidden=G_HIDDEN, out_dim=IMAGE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(hidden, hidden * 2),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(hidden * 2, hidden * 4),
            nn.BatchNorm1d(hidden * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(hidden * 4, out_dim),
            nn.Tanh(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """
    Fully-connected GAN discriminator.

    784-dim image -> (Linear + LeakyReLU + Dropout) x 3 -> Linear -> Sigmoid
    Output: probability in (0, 1) that the input is a real image.

    No BatchNorm (empirically more stable in the discriminator).
    """
    def __init__(self, in_dim=IMAGE_DIM, hidden=D_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(hidden * 4, hidden * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


G = Generator().to(device)
D = Discriminator().to(device)

print("Generator architecture:")
print(G)
print(f"\nGenerator parameters  : {sum(p.numel() for p in G.parameters()):,}")
print("\nDiscriminator architecture:")
print(D)
print(f"\nDiscriminator parameters: {sum(p.numel() for p in D.parameters()):,}")


### 3.9  MNIST dataset

Pixels normalised to $[-1,1]$ via Normalize$(\mu=0.5,\sigma=0.5)$,
matching the Tanh output range of $G$.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                          # [0, 1]
    transforms.Normalize(mean=(0.5,), std=(0.5,)), # -> [-1, 1]
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, drop_last=True
)

print(f"Training samples : {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")

x_sample, y_sample = next(iter(train_loader))
print(f"Batch shape      : {x_sample.shape}  (B, C, H, W)")
print(f"Pixel range      : [{x_sample.min():.2f}, {x_sample.max():.2f}]")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for k, ax in enumerate(axes.flat):
    img = x_sample[k, 0].numpy()
    ax.imshow(img, cmap='gray', vmin=-1, vmax=1)
    ax.set_title(str(y_sample[k].item()), fontsize=9)
    ax.axis('off')
plt.suptitle('Real MNIST samples (normalised to [-1, 1])', y=1.01)
plt.tight_layout()
plt.show()


### 3.10  Loss functions and noise sampler

**One-sided label smoothing** (real targets $= 0.9$ instead of $1$):
prevents $D$ from becoming overconfident, providing softer gradients to $G$.


In [ ]:
def sample_noise(batch_size, z_dim=Z_DIM, mode=MODE_Z):
    """
    Sample latent vectors z from the prior p_z.
    mode='normal'  -> z ~ N(0, I)   (standard choice)
    mode='uniform' -> z ~ Uniform(-1, 1)
    """
    if mode == 'normal':
        return torch.randn(batch_size, z_dim, device=device)
    elif mode == 'uniform':
        return torch.rand(batch_size, z_dim, device=device) * 2 - 1
    else:
        raise ValueError(f"Unknown noise mode: {mode}")


bce_loss = nn.BCELoss()


def d_loss_fn(real_preds, fake_preds):
    """
    Discriminator loss with one-sided label smoothing:
        L_D = BCE(D(x), 0.9) + BCE(D(G(z)), 0)
    Label smoothing on real targets prevents over-confident discriminators
    and provides softer gradients to the generator.
    """
    B = real_preds.size(0)
    real_labels = torch.full((B, 1), LABEL_SMOOTH, device=device)
    fake_labels = torch.zeros(B, 1, device=device)
    return bce_loss(real_preds, real_labels) + bce_loss(fake_preds, fake_labels)


def g_loss_fn(fake_preds):
    """
    Non-saturating generator loss:
        L_G = BCE(D(G(z)), 1) = -E[log D(G(z))]

    This has a strong gradient even when D(G(z)) is near 0,
    unlike the original loss log(1 - D(G(z))) which saturates.
    """
    B = fake_preds.size(0)
    real_labels = torch.ones(B, 1, device=device)
    return bce_loss(fake_preds, real_labels)


# Sanity check before training
z_test = sample_noise(4)
x_fake = G(z_test)
x_real_batch, _ = next(iter(train_loader))
x_real_batch = x_real_batch.view(BATCH_SIZE, -1).to(device)

with torch.no_grad():
    dp = D(x_real_batch[:4])
    df = D(x_fake)

print(f"D(real) before training: {dp.squeeze().tolist()}")
print(f"D(fake) before training: {df.squeeze().tolist()}")
print(f"D loss  before training: {d_loss_fn(dp, df).item():.4f}")
print(f"G loss  before training: {g_loss_fn(df).item():.4f}")


### 3.11  Optimisers

Adam with $\beta_1=0.5$ (lower than default 0.9): reduces momentum in
the non-stationary GAN landscape where both networks evolve simultaneously.


In [ ]:
# Adam with beta_1=0.5: reduces momentum in the non-stationary GAN landscape
g_opt = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(BETA1, 0.999))
d_opt = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(BETA1, 0.999))

# Fixed noise for consistent visualisation across epochs
fixed_z = sample_noise(64)
print(f"Fixed noise shape: {fixed_z.shape}")
print(f"Optimisers ready. Training for {NUM_EPOCHS} epochs...")


### 3.12  Training loop

Per-batch procedure:
1. **Discriminator step**: forward on real (with `.detach()` on fake) $\to$ $\mathcal{L}_D$ $\to$ update $D$ only.
2. **Generator step**: forward on fresh fake (no `.detach()`) $\to$ $\mathcal{L}_G$ $\to$ update $G$ only.


In [ ]:
d_losses, g_losses         = [], []
d_real_probs, d_fake_probs = [], []
epoch_images               = []

print(f"{'Epoch':>5}  {'D loss':>8}  {'G loss':>8}  {'D(real)':>8}  {'D(fake)':>8}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    G.train(); D.train()
    d_loss_sum = g_loss_sum = d_real_sum = d_fake_sum = 0.0
    n_batches = 0

    for real_imgs, _ in train_loader:
        B = real_imgs.size(0)
        real_imgs = real_imgs.view(B, -1).to(device)

        # ── Discriminator step ────────────────────────────────────────
        d_opt.zero_grad()
        real_preds = D(real_imgs)

        z = sample_noise(B)
        fake_imgs  = G(z).detach()    # detach: no gradient through G
        fake_preds = D(fake_imgs)

        d_loss = d_loss_fn(real_preds, fake_preds)
        d_loss.backward()
        d_opt.step()

        # ── Generator step ────────────────────────────────────────────
        g_opt.zero_grad()
        z = sample_noise(B)
        fake_imgs_g  = G(z)
        fake_preds_g = D(fake_imgs_g)   # do NOT detach here

        g_loss = g_loss_fn(fake_preds_g)
        g_loss.backward()
        g_opt.step()

        d_loss_sum += d_loss.item()
        g_loss_sum += g_loss.item()
        d_real_sum += real_preds.mean().item()
        d_fake_sum += fake_preds.mean().item()
        n_batches  += 1

    d_losses.append(d_loss_sum / n_batches)
    g_losses.append(g_loss_sum / n_batches)
    d_real_probs.append(d_real_sum / n_batches)
    d_fake_probs.append(d_fake_sum / n_batches)

    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z).view(64, 1, 28, 28).cpu()
        imgs = (imgs + 1) / 2.0
    epoch_images.append(imgs)

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>5}  "
              f"{d_losses[-1]:>8.4f}  "
              f"{g_losses[-1]:>8.4f}  "
              f"{d_real_probs[-1]:>8.4f}  "
              f"{d_fake_probs[-1]:>8.4f}")


### 3.13  Learning curves

At the Nash equilibrium $D(\mathbf{x})$ and $D(G(\mathbf{z}))$ both converge to $0.5$,
and the discriminator loss stabilises near $2\log 2 \approx 1.386$.


In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, d_losses, lw=2, label='Discriminator loss')
axes[0].plot(epochs, g_losses, lw=2, label='Generator loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training losses')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, d_real_probs, lw=2, label=r'$D(\mathbf{x})$  real')
axes[1].plot(epochs, d_fake_probs, lw=2, ls='--', label=r'$D(G(\mathbf{z}))$  fake')
axes[1].axhline(0.5, color='k', ls=':', lw=1.2, label='Nash eq. (0.5)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Mean discriminator output')
axes[1].set_title('Discriminator outputs — converging to 0.5')
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 1)

plt.tight_layout(); plt.show()
print(f"Final D(real): {d_real_probs[-1]:.4f}   Nash equilibrium target: 0.5")
print(f"Final D(fake): {d_fake_probs[-1]:.4f}   Nash equilibrium target: 0.5")


### 3.14  Generated images over training

In [ ]:
selected_epochs = [1, 5, 10, 20, 35, NUM_EPOCHS]
selected_epochs = sorted(set(e for e in selected_epochs if e <= NUM_EPOCHS))

n_cols = 5
fig, axes = plt.subplots(len(selected_epochs), n_cols,
                         figsize=(n_cols * 1.8, len(selected_epochs) * 2.0))

for row, epoch in enumerate(selected_epochs):
    imgs = epoch_images[epoch - 1]
    for col in range(n_cols):
        ax = axes[row, col]
        ax.imshow(imgs[col, 0].numpy(), cmap='gray_r', vmin=0, vmax=1)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(f'Epoch {epoch}', rotation=0, labelpad=48,
                          fontsize=10, color='firebrick', va='center')

plt.suptitle('Generator output at selected epochs', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


### 3.15  Final generated samples vs real MNIST

In [ ]:
final_imgs = epoch_images[-1]   # (64, 1, 28, 28) in [0, 1]
real_cmp, _ = next(iter(train_loader))

fig, axes = plt.subplots(4, 16, figsize=(16, 5))
for col in range(16):
    for row in range(2):
        ax = axes[row, col]
        ax.imshow(real_cmp[row * 16 + col, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
        ax.axis('off')
    for row in range(2):
        ax = axes[row + 2, col]
        ax.imshow(final_imgs[row * 16 + col, 0].numpy(), cmap='gray_r', vmin=0, vmax=1)
        ax.axis('off')

axes[0, 0].set_ylabel('Real',      rotation=0, labelpad=38, fontsize=11, va='center')
axes[2, 0].set_ylabel('Generated', rotation=0, labelpad=55, fontsize=11,
                       va='center', color='steelblue')
plt.suptitle('Real MNIST (top rows) vs Generated digits (bottom rows)', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()


### 3.16  MMD quality metric

In [ ]:
def pairwise_sq_dist(X, Y):
    """
    Pairwise squared Euclidean distances.
    Uses the identity ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a^T*b.
    Returns shape (nX, nY).
    """
    nX, nY   = X.size(0), Y.size(0)
    X_f      = X.view(nX, -1).float()
    Y_f      = Y.view(nY, -1).float()
    X2       = (X_f ** 2).sum(1, keepdim=True)
    Y2       = (Y_f ** 2).sum(1, keepdim=True).t()
    return (X2 + Y2 - 2 * torch.mm(X_f, Y_f.t())).clamp(min=0.0)


def gaussian_kernel(M, sigma=1.0):
    """k(x,y) = exp(-||x-y||^2 / (2 * scale * sigma^2))."""
    scale = M.mean().clamp(min=1e-8)
    return torch.exp(-M / (2.0 * scale * sigma ** 2))


def mmd_score(real_imgs, fake_imgs, sigma=1.0):
    """
    Unbiased MMD estimator between real and fake image batches.
    MMD = 0 iff p_r = p_g (for a characteristic kernel).
    Lower is better.
    """
    Mxx = pairwise_sq_dist(real_imgs, real_imgs)
    Mxy = pairwise_sq_dist(real_imgs, fake_imgs)
    Myy = pairwise_sq_dist(fake_imgs, fake_imgs)

    Kxx = gaussian_kernel(Mxx, sigma)
    Kxy = gaussian_kernel(Mxy, sigma)
    Kyy = gaussian_kernel(Myy, sigma)

    n, m = Kxx.size(0), Kyy.size(0)
    Kxx.fill_diagonal_(0)
    Kyy.fill_diagonal_(0)
    mmd2 = (Kxx.sum() / (n * (n - 1))
           + Kyy.sum() / (m * (m - 1))
           - 2 * Kxy.mean())
    return float(mmd2.clamp(min=0).sqrt())


# Evaluate
N_EVAL = 1000
eval_loader = DataLoader(train_dataset, batch_size=N_EVAL, shuffle=True)
real_eval, _ = next(iter(eval_loader))
real_eval = real_eval.view(N_EVAL, -1)

G.eval()
with torch.no_grad():
    z_eval = sample_noise(N_EVAL)
    fake_eval = G(z_eval).cpu()

score = mmd_score(real_eval, fake_eval)
print(f"MMD (real vs generated): {score:.5f}   (lower = better; 0 = identical distributions)")

# Reference: MMD between two disjoint real subsets
real_eval2, _ = next(iter(eval_loader))
real_eval2 = real_eval2.view(N_EVAL, -1)
score_ref = mmd_score(real_eval, real_eval2)
print(f"MMD (real vs real):      {score_ref:.5f}   (reference baseline)")


### 3.17  Latent space interpolation

In [ ]:
G.eval()
n_interp, n_pairs = 10, 4

fig, axes = plt.subplots(n_pairs, n_interp,
                         figsize=(n_interp * 1.5, n_pairs * 1.8))
alphas = torch.linspace(0, 1, n_interp)

with torch.no_grad():
    for row in range(n_pairs):
        z1 = sample_noise(1)
        z2 = sample_noise(1)
        for col, alpha in enumerate(alphas):
            z = (1 - alpha) * z1 + alpha * z2
            img = (G(z).view(28, 28).cpu() + 1) / 2.0
            axes[row, col].imshow(img.numpy(), cmap='gray_r', vmin=0, vmax=1)
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(f'Pair {row+1}', fontsize=9,
                                va='center', rotation=0, labelpad=40)

for col, a in enumerate(alphas.tolist()):
    axes[0, col].set_title(f'alpha={a:.1f}', fontsize=8)

plt.suptitle('Latent space interpolation: smooth transitions between digits',
             fontsize=11, y=1.01)
plt.tight_layout(); plt.show()


---
## Section 4 — Comparing Deep Generative Model Families

This section provides a systematic comparison of the four main families of
deep generative models used in modern machine learning and scientific computing:
**(1)** Energy-Based Models (including Restricted Boltzmann Machines),
**(2)** Variational Autoencoders,
**(3)** Diffusion Models, and
**(4)** Generative Adversarial Networks.

### 4.1  Unified probabilistic view

Every generative model defines a distribution $p_{\boldsymbol{\theta}}(\mathbf{x})$.
They differ in how they parameterise and train it:

$$
\begin{array}{lccc}
\textbf{Model} & \textbf{Parameterises} & \textbf{Training signal} & \textbf{Sampling}\\
\hline
\text{EBM/RBM} & E_{\boldsymbol{\theta}}(\mathbf{x}) & \text{Contrastive Divergence} & \text{MCMC}\\
\text{VAE} & p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{h}),q_{\boldsymbol{\phi}}(\mathbf{h}|\mathbf{x}) & \text{ELBO (variational)} & \text{1 decoder pass}\\
\text{Diffusion} & \boldsymbol{\epsilon}_{\boldsymbol{\theta}}(\mathbf{x}_t,t) & \text{ELBO (hierarchical)} & T\text{ denoiser passes}\\
\text{GAN} & G_{\boldsymbol{\theta}}(\mathbf{z}), D_{\boldsymbol{\phi}}(\mathbf{x}) & \text{Adversarial (no likelihood)} & \text{1 generator pass}
\end{array}
$$

### 4.2  Energy-Based Models and Restricted Boltzmann Machines

#### Energy function and Boltzmann distribution

An **Energy-Based Model (EBM)** defines the distribution through an energy function
$E_{\boldsymbol{\theta}}(\mathbf{x})$:

$$p_{\boldsymbol{\theta}}(\mathbf{x})
= \frac{\exp(-E_{\boldsymbol{\theta}}(\mathbf{x}))}{Z(\boldsymbol{\theta})},
\qquad Z(\boldsymbol{\theta}) = \int \exp(-E_{\boldsymbol{\theta}}(\mathbf{x}))\,d\mathbf{x}.$$

The partition function $Z$ is generally intractable.

#### Restricted Boltzmann Machine

An **RBM** is a bipartite undirected graphical model with visible units $\mathbf{v}$
and hidden units $\mathbf{h}$ (binary). The energy is:

$$E(\mathbf{v},\mathbf{h}) = -\mathbf{a}^\top\mathbf{v} - \mathbf{b}^\top\mathbf{h}
- \mathbf{v}^\top W\mathbf{h},$$

giving the joint $p(\mathbf{v},\mathbf{h})\propto\exp(-E(\mathbf{v},\mathbf{h}))$.

Because there are no intra-layer connections, the conditional distributions
factorise:

$$p(h_j=1|\mathbf{v}) = \sigma\!\left(b_j + \sum_i W_{ij}v_i\right),
\qquad
p(v_i=1|\mathbf{h}) = \sigma\!\left(a_i + \sum_j W_{ij}h_j\right).$$

#### Contrastive Divergence (CD-$k$) training

The log-likelihood gradient is:

$$\nabla_{\boldsymbol{\theta}}\log p(\mathbf{v})
= -\mathbb{E}_{p(\mathbf{h}|\mathbf{v})}\!\left[\nabla E\right]
+ \mathbb{E}_{p(\mathbf{v},\mathbf{h})}\!\left[\nabla E\right].$$

The first term (positive phase) is tractable; the second (negative phase, model
expectation) requires MCMC. CD-$k$ approximates the negative phase by running
$k$ steps of Gibbs sampling from the data, not from the model prior.

#### Deep Boltzmann Machines and Belief Networks

Stacking RBMs gives **Deep Belief Networks (DBN)** or, if connections are
undirected at every level, **Deep Boltzmann Machines (DBM)**. These can model
hierarchical features but training becomes progressively harder due to:
- The partition function growing exponentially with depth
- Slow mixing of MCMC chains in the negative phase
- Layer-by-layer greedy pre-training required for stability

### 4.3  VAE vs Diffusion — the ELBO hierarchy

Both models optimise the same variational lower bound:

$$\log p(\mathbf{x}) \geq \text{ELBO}(\boldsymbol{\phi},\boldsymbol{\theta}).$$

Their difference is the **number of latent levels**:

| Aspect | VAE | DDPM |
|--------|-----|------|
| Latent levels | 1 (bottleneck $\mathbf{h}\in\mathbb{R}^{d_h}$, $d_h\ll d_x$) | $T=300$–$1000$ (full-dim $\mathbf{x}_t$) |
| Encoder $q$ | Learned: $q_{\boldsymbol{\phi}}(\mathbf{h}|\mathbf{x})$ | Fixed Gaussian schedule |
| Reparameterisation | $\mathbf{h}=\boldsymbol{\mu}+\boldsymbol{\sigma}\odot\boldsymbol{\epsilon}$ | $\mathbf{x}_t=\sqrt{\bar\alpha_t}\mathbf{x}_0+\sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}$ |
| Training loss | $\text{BCE}+\beta\,D_{\mathrm{KL}}$ (1 term per sample) | $\mathbb{E}\|\boldsymbol{\epsilon}-\hat{\boldsymbol{\epsilon}}_{\boldsymbol{\theta}}\|^2$ ($T$ terms averaged) |
| Sample quality | Blurry (posterior averaging) | Sharp (iterative refinement) |
| Latent structure | $\checkmark$ (interpolation, clustering) | $\times$ (no compact latent) |
| Sampling cost | 1 decoder call | $T$ denoiser calls |

**Why VAE samples are blurry.** The Bernoulli/Gaussian decoder minimises a
pixel-wise reconstruction loss *averaged over the posterior*. Because the
posterior $q_{\boldsymbol{\phi}}(\mathbf{h}|\mathbf{x})$ has nonzero variance,
the decoder learns the mean over all plausible completions — which is blurry.
This is a fundamental limitation, not fixable by deeper networks alone.

**Why DDPM samples are sharp.** DDPM never predicts $\mathbf{x}_0$ in a
single shot. Each reverse step corrects only a small amount of noise,
allowing the network to commit to fine details incrementally without
averaging over the full distribution of plausible outputs.

**Posterior collapse in VAEs.** A powerful decoder can reconstruct $\mathbf{x}$
without using $\mathbf{h}$, making $q_{\boldsymbol{\phi}}(\mathbf{h}|\mathbf{x})\approx p(\mathbf{h})$,
the KL drops to zero, and the latent space becomes useless. DDPM avoids this
because the fixed forward process guarantees the reverse network must use
the noisy input $\mathbf{x}_t$ at each step.

### 4.4  GAN vs likelihood-based models

GANs are the only family here that do **not** optimise a likelihood bound.
This has profound consequences:

**Advantages of GANs:**
- Can produce extremely sharp, high-frequency detail (no blurring from BCE/MSE loss)
- Single generator pass at inference (fastest of all four families)
- Naturally supports conditional generation via label concatenation
- Training signal can be made adversarial even when the true likelihood is intractable

**Disadvantages of GANs:**
- **No likelihood estimate**: cannot compute $p(\mathbf{x})$ or detect anomalies
- **Mode collapse**: generator may ignore large parts of $p_r$
- **Training instability**: non-stationary landscape; simultaneous updates
  do not guarantee convergence to Nash equilibrium
- **JS divergence is flat** when $p_r$ and $p_g$ have non-overlapping supports
  (common early in training, unlike Wasserstein distance which is always informative)

**Why GAN samples are sharp.** The adversarial discriminator acts as an
adaptive, learned perceptual metric. It penalises any systematic deviation from
real images — including blurriness — more severely than a pixel-wise MSE loss would.

### 4.5  Comprehensive comparison table

| Criterion | RBM/EBM | VAE | Diffusion | GAN |
|-----------|---------|-----|-----------|-----|
| **Likelihood estimate** | $\approx$ (AIS) | $\checkmark$ (ELBO lower bound) | $\checkmark$ (ELBO) | $\times$ |
| **Sample quality** | Low–Medium | Medium (blurry) | High (state of art) | High (sharp) |
| **Sampling speed** | Slow (MCMC) | Fast (1 pass) | Slow ($T$ passes) | Fast (1 pass) |
| **Training stability** | Moderate (CD) | High | High | Low–Moderate |
| **Structured latent** | Partially (hidden units) | $\checkmark$ | $\times$ | $\times$ |
| **Mode coverage** | Partial | Good | Excellent | Partial (collapse) |
| **Conditional gen.** | Hard | Moderate | Easy | Easy (cGAN) |
| **Anomaly detection** | $\checkmark$ (energy score) | $\checkmark$ (ELBO score) | $\checkmark$ | $\times$ |
| **Theoretical grounding** | Statistical mechanics | Variational inference | Stochastic processes | Game theory |
| **Scales to high-res** | No | With LDM | Yes (LDM) | Yes (DCGAN, StyleGAN) |
| **Scientific applications** | Molecular design (RBM) | Representation learning | Inverse problems | Data augmentation |

### 4.6  Pros and cons in detail

#### Energy-Based Models / RBMs

**Pros:**
- Elegant probabilistic framework rooted in statistical mechanics
- Undirected model: symmetric treatment of visible and hidden variables
- Principled anomaly detection via the energy score $E_{\boldsymbol{\theta}}(\mathbf{x})$
- Natural connection to physics (Ising models, Hopfield networks)
- Good inductive bias for structured discrete data (spin systems, binary variables)

**Cons:**
- Partition function $Z$ is intractable for all but tiny models
- MCMC/CD training is slow and does not scale to deep architectures
- Sample quality is far below VAE/diffusion/GAN for natural images
- Mixing times of Gibbs chains can be exponentially long in the model dimension
- Stacking layers (DBM) requires layer-wise greedy pre-training; hard to end-to-end optimise

**When to use:** physics-inspired problems (lattice models, binary data),
anomaly detection where an energy score is interpretable, small-scale problems
where MCMC is feasible, and as a building block inside hybrid models.

---

#### Variational Autoencoders

**Pros:**
- Tractable lower bound on $\log p(\mathbf{x})$: principled likelihood evaluation
- Compact, structured latent space: interpolation, clustering, property prediction
- Single encoder+decoder pass: fast inference and generation
- $\beta$-VAE variant promotes disentanglement for interpretability
- Naturally extendable to semi-supervised and hierarchical settings
- Stable training: ELBO is a well-defined objective with no adversarial dynamics

**Cons:**
- Blurry samples due to posterior averaging in the decoder
- ELBO gap: $\log p(\mathbf{x}) = \text{ELBO} + D_{\mathrm{KL}}(q_{\boldsymbol{\phi}}\|p(\mathbf{h}|\mathbf{x}))$
  and the gap is positive unless the mean-field Gaussian exactly matches the true posterior
- Posterior collapse: powerful decoders can ignore the latent space
- Mean-field approximation: $q_{\boldsymbol{\phi}}$ factorises across latent dimensions,
  which may miss correlations in the true posterior

**When to use:** representation learning, anomaly detection, scientific data
(molecules, spectra) where a compact latent enables property prediction,
real-time generation where a single decoder pass is required, and as the
compression stage in Latent Diffusion Models.

---

#### Diffusion Models (DDPM/DDIM/Score-based)

**Pros:**
- State-of-the-art sample quality across images, audio, and video
- No mode collapse: the reverse process covers the full support of $p_r$
- Tractable ELBO: approaches $\log p(\mathbf{x})$ as $T\to\infty$ with a fine schedule
- Flexible conditioning: classifier-free guidance enables text/image conditioning
- Connection to score matching and stochastic differential equations (SDEs)
  gives a rich theoretical framework (Song et al., NeurIPS 2020)
- Training is stable: no adversarial dynamics

**Cons:**
- Slow sampling: $T=300$–$1000$ forward passes at inference
  (DDIM and consistency models reduce this to $\sim 10$–$50$ steps)
- No compact latent space: $\mathbf{x}_t$ lives in the same space as $\mathbf{x}_0$
- Computationally expensive to train at high resolution
  (Latent Diffusion sidesteps this by working in a VAE's latent space)
- Less sample diversity at very low inference budgets (few steps)

**When to use:** high-fidelity image/audio/video synthesis, text-to-image
generation, inverse problems (inpainting, super-resolution) where the score
function provides principled regularisation, and scientific simulation
(protein structure, molecular dynamics) where quality matters more than speed.

---

#### Generative Adversarial Networks

**Pros:**
- Sharpest samples: the adversarial discriminator penalises perceptual artefacts
  more effectively than any fixed pixel-wise loss
- Fastest inference: single generator pass
- Flexible loss: adversarial training works even when the true likelihood is intractable
- Rich family of extensions: cGAN, WGAN, DCGAN, StyleGAN, CycleGAN, etc.
- State-of-the-art for image-to-image translation and style transfer

**Cons:**
- No likelihood estimate: cannot evaluate $p(\mathbf{x})$ for anomaly detection or density estimation
- Mode collapse: generator learns a subset of modes and ignores the rest
- Training instability: simultaneous gradient updates on $G$ and $D$ do not
  guarantee convergence; hyperparameter sensitivity is high
- JS divergence is flat when supports do not overlap (WGAN addresses this)
- The discriminator is discarded after training — wasted capacity

**When to use:** high-quality image synthesis where speed matters more than
diversity, image-to-image translation (pix2pix, CycleGAN), super-resolution,
data augmentation where sharp artefact-free samples are required, and any
setting where an explicit likelihood is not needed.

### 4.7  The Latent Diffusion Model: best of both worlds

**Latent Diffusion Models** (LDM; Rombach et al. 2022, the architecture behind
Stable Diffusion) elegantly combine VAE and DDPM:

1. **VAE compression** maps $\mathbf{x}\in\mathbb{R}^{H\times W\times 3}$ to a
   compact latent $\mathbf{h}\in\mathbb{R}^{h/8\times w/8\times 4}$ — a $48\times$
   reduction in dimension.
2. **DDPM in latent space** runs the full diffusion process on $\mathbf{h}$,
   not on the pixel space.
3. **VAE decoder** maps the denoised latent back to pixel space.

Benefits: DDPM's sample quality at the cost of DDPM in a much smaller space;
VAE's compact latent enabling text conditioning via cross-attention.


### 4.8  Quantitative comparison: VAE vs DDPM vs GAN

In [ ]:
# ── Helper: pixel-space FID proxy ─────────────────────────────────────────
# True FID uses Inception features and is expensive.
# We use a simpler proxy: compare covariance of flattened pixel intensities.
def pixel_stats(loader_or_tensor, n_max=2000):
    """Return mean and covariance of flattened pixel arrays (proxy for FID)."""
    if isinstance(loader_or_tensor, torch.Tensor):
        imgs = loader_or_tensor[:n_max].view(min(len(loader_or_tensor), n_max), -1).numpy()
    else:
        chunks = []
        for x, _ in loader_or_tensor:
            chunks.append(x.view(x.size(0), -1))
            if sum(c.size(0) for c in chunks) >= n_max:
                break
        imgs = torch.cat(chunks)[:n_max].numpy()
    mu  = imgs.mean(axis=0)
    cov = np.cov(imgs.T)
    return mu, cov


def proxy_fid(mu_r, cov_r, mu_g, cov_g):
    """
    Frechet distance between two Gaussians:
        FD = ||mu_r - mu_g||^2 + Tr(cov_r + cov_g - 2*sqrt(cov_r @ cov_g))
    We use only the diagonal (pixel variances) for speed.
    """
    diff = np.sum((mu_r - mu_g)**2)
    # Diagonal approximation: sqrt(diag(cov_r) * diag(cov_g))
    diag_r = np.diag(cov_r)
    diag_g = np.diag(cov_g)
    tr_term = np.sum(diag_r + diag_g - 2.0*np.sqrt(np.maximum(diag_r*diag_g, 0)))
    return diff + tr_term


# Collect real test images
mu_real, cov_real = pixel_stats(loader_test_01, n_max=2000)

# VAE samples (already generated above, extend to 2000)
vae.eval()
with torch.no_grad():
    z_big   = torch.randn(2000, vae.latent_dim, device=device)
    vae_big = torch.sigmoid(vae.decode(z_big)).cpu()
mu_vae, cov_vae = pixel_stats(vae_big, n_max=2000)

# DDPM samples (generate 256 for speed)
print('Generating 256 DDPM samples for quantitative comparison...')
ddpm_model.eval()
ddpm_big = []
with torch.no_grad():
    for _ in range(256 // 16):
        s = ddpm_sample(ddpm_model, n_samples=16, verbose=False)
        ddpm_big.append((s * 0.5 + 0.5).cpu())
ddpm_big = torch.cat(ddpm_big)
mu_ddpm, cov_ddpm = pixel_stats(ddpm_big, n_max=256)

fid_vae  = proxy_fid(mu_real, cov_real, mu_vae,  cov_vae)
fid_ddpm = proxy_fid(mu_real, cov_real, mu_ddpm, cov_ddpm)

# Sharpness proxy: variance of Laplacian (higher = sharper)
def laplacian_sharpness(tensor_01):
    """Mean sharpness of a batch of [0,1] images."""
    import torch.nn.functional as F2
    ker = torch.tensor([[0,-1,0],[-1,4,-1],[0,-1,0]],
                       dtype=torch.float32).view(1,1,3,3)
    laps = F2.conv2d(tensor_01, ker, padding=1)
    return float(laps.var())


sharpness_real = laplacian_sharpness(next(iter(loader_test_01))[0][:256])
sharpness_vae  = laplacian_sharpness(vae_big[:256])
sharpness_ddpm = laplacian_sharpness(ddpm_big[:256])

print()
print(f'{'Model':<18s}  {'Proxy-FID':>10s}  {'Sharpness':>10s}  {'Train time':>12s}  {'Sample time':>13s}')
print('-' * 70)
print(f'{'Real data':<18s}  {'—':>10s}  {sharpness_real:>10.4f}  {'—':>12s}  {'—':>13s}')
print(f'{'VAE':<18s}  {fid_vae:>10.2f}  {sharpness_vae:>10.4f}  '
      f'{vae_history["wall_s"]:>11.1f}s  {"~0.01s/img":>13s}')
print(f'{'DDPM':<18s}  {fid_ddpm:>10.2f}  {sharpness_ddpm:>10.4f}  '
      f'{ddpm_history["wall_s"]:>11.1f}s  {f"~{T*0.002:.1f}s/img":>13s}')


### 4.9  Visual comparison: VAE, DDPM, and GAN side by side

In [ ]:
# ── Side-by-side sample grid ───────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, hspace=0.35, wspace=0.25)

# ── Row 0: Real, VAE, DDPM grids ──────────────────────────────────────────
for col, (title, imgs) in enumerate([
    ('Real MNIST (test set)',   next(iter(loader_test_01))[0][:16]),
    ('VAE samples',             vae_big[:16]),
    ('DDPM samples',            ddpm_big[:16]),
]):
    ax  = fig.add_subplot(gs[0, col])
    grid = make_grid(imgs.clamp(0, 1), nrow=4, normalize=False, pad_value=0.9)
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

# ── Row 1: Training curves ─────────────────────────────────────────────────
ax_vae = fig.add_subplot(gs[1, 0])
ep_v   = range(1, len(vae_history['elbo']) + 1)
ax_vae.plot(ep_v, vae_history['elbo'],  'b-',  lw=2, label='ELBO')
ax_vae.plot(ep_v, vae_history['recon'], 'b--', lw=1.5, label='Reconstruction')
ax_vae.plot(ep_v, vae_history['kl'],    'b:',  lw=1.5, label='KL')
ax_vae.set_title('VAE Training Curves'); ax_vae.set_xlabel('Epoch')
ax_vae.legend(fontsize=9); ax_vae.grid(True, alpha=0.3)

ax_ddpm = fig.add_subplot(gs[1, 1])
ep_d    = range(1, len(ddpm_history['loss']) + 1)
ax_ddpm.plot(ep_d, ddpm_history['loss'], 'r-', lw=2)
ax_ddpm.set_title('DDPM Training: Noise MSE'); ax_ddpm.set_xlabel('Epoch')
ax_ddpm.grid(True, alpha=0.3)

# ── Row 1 col 2: power spectrum (frequency content = sharpness indicator) ──
ax_ps = fig.add_subplot(gs[1, 2])

def mean_power_spectrum(imgs_01, label, color, ax):
    ps = []
    for img in imgs_01[:64]:
        f  = np.fft.fft2(img.squeeze().numpy())
        ps.append(np.abs(np.fft.fftshift(f))**2)
    mean_ps = np.stack(ps).mean(0)
    # Radially averaged profile
    H, W = mean_ps.shape
    cy, cx = H//2, W//2
    rs = np.sqrt(((np.arange(H)-cy)[:,None])**2 +
                 ((np.arange(W)-cx)[None,:])**2).astype(int)
    profile = np.array([mean_ps[rs==r].mean() if (rs==r).any() else 0
                        for r in range(min(cy,cx))])
    ax.semilogy(profile[1:], label=label, color=color, lw=2)

mean_power_spectrum(next(iter(loader_test_01))[0][:64],
                    'Real', 'black', ax_ps)
mean_power_spectrum(vae_big[:64],  'VAE',  'steelblue', ax_ps)
mean_power_spectrum(ddpm_big[:64], 'DDPM', 'firebrick',  ax_ps)
ax_ps.set_title('Mean Power Spectrum (sharpness)')
ax_ps.set_xlabel('Spatial frequency'); ax_ps.set_ylabel('Power')
ax_ps.legend(); ax_ps.grid(True, alpha=0.3)

# ── Row 2: Summary table ───────────────────────────────────────────────────
ax_tab = fig.add_subplot(gs[2, :])
ax_tab.axis('off')

table_data = [
    ['', 'VAE', 'DDPM', 'Notes'],
    ['Parameters', f'{n_params_vae:,}', f'{n_params_ddpm:,}', ''],
    ['Latent dim', str(vae.latent_dim), f'{IMG_SIZE*IMG_SIZE}={IMG_SIZE**2}', 'DDPM: full image dim'],
    ['Inference', '1 encoder pass', '1 forward pass', 'Same cost'],
    ['Sampling', '1 decoder pass', f'{T} denoiser passes', f'DDPM is ~{T}x slower'],
    ['Train loss', 'ELBO (2 terms)', 'Noise MSE (1 term)', ''],
    ['Proxy FID', f'{fid_vae:.1f}', f'{fid_ddpm:.1f}', 'Lower is better'],
    ['Sharpness', f'{sharpness_vae:.4f}', f'{sharpness_ddpm:.4f}', 'Higher is better'],
    ['Latent structure', 'Yes (see 2-D plot)', 'No', ''],
    ['Mode collapse', 'Rare', 'Not observed', ''],
]

tbl = ax_tab.table(
    cellText   = table_data[1:],
    colLabels  = table_data[0],
    loc        = 'center',
    cellLoc    = 'left',
)
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
tbl.scale(1, 1.6)
ax_tab.set_title('Quantitative Summary', fontsize=12, fontweight='bold', pad=8)

plt.suptitle('VAE vs DDPM on MNIST — Comprehensive Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('vae_vs_ddpm.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figure saved to vae_vs_ddpm.png')


---
## Section 5 — Exercises

### Mathematical exercises

#### VAE / ELBO

1. **ELBO derivation.**
   Starting from Jensen's inequality applied to $\log p(\mathbf{x})$,
   derive the ELBO $\mathcal{L}(\boldsymbol{\phi},\boldsymbol{\theta})$ and
   show that $\log p(\mathbf{x}) - \mathcal{L} = D_{\mathrm{KL}}(q_{\boldsymbol{\phi}}\|p(\mathbf{h}|\mathbf{x}))\geq 0$.

2. **Closed-form KL.**
   Verify
   $D_{\mathrm{KL}}(\mathcal{N}(\boldsymbol{\mu},\mathrm{diag}(\boldsymbol{\sigma}^2))\,\|\,\mathcal{N}(\mathbf{0},\mathbf{I}))
   = \frac{1}{2}\sum_j(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1)$
   and identify which term prevents posterior collapse.

#### DDPM

3. **Forward marginal (induction).**
   Starting from $q(\mathbf{x}_t|\mathbf{x}_{t-1})=\mathcal{N}(\sqrt{\alpha_t}\mathbf{x}_{t-1},(1-\alpha_t)\mathbf{I})$,
   prove by induction that
   $q(\mathbf{x}_t|\mathbf{x}_0)=\mathcal{N}(\sqrt{\bar\alpha_t}\mathbf{x}_0,(1-\bar\alpha_t)\mathbf{I})$.

4. **Tractable forward posterior.**
   Using Bayes' rule, derive
   $q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0)$ and show it is Gaussian with mean
   $\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}\mathbf{x}_0
   + \frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\mathbf{x}_t$
   and variance $\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$.

5. **ELBO decomposition.**
   Show that the DDPM ELBO decomposes into
   $\mathcal{L}_0 + \mathcal{L}_T + \sum_{t=2}^{T}\mathcal{L}_{t-1}$
   and explain why $\mathcal{L}_T\approx 0$ by the design of the noise schedule.

#### GAN

6. **Optimal discriminator.**
   Derive $D^*(\mathbf{x})=p_r/(p_r+p_g)$ by differentiating
   $f(t)=A\log t+B\log(1-t)$ and verify it is a maximum via $f''(t^*)<0$.

7. **GAN = JS divergence.**
   Expand $D_{\mathrm{JS}}(p_r\|p_g)$ and show that
   $V(G,D^*)=2D_{\mathrm{JS}}(p_r\|p_g)-2\log 2$.

8. **Vanishing gradient.**
   At $D(G(\mathbf{z}))=\varepsilon\approx 0$, compute
   $|\nabla_{\boldsymbol{\theta}^{(g)}}\mathcal{L}_G^{\text{orig}}|$ and
   $|\nabla_{\boldsymbol{\theta}^{(g)}}\mathcal{L}_G^{\text{non-sat}}|$.
   Show the non-saturating form diverges as $\varepsilon\to 0$.

#### Energy-Based Models

9. **RBM conditional independence.**
   Show that because the RBM has no intra-layer connections,
   $p(\mathbf{h}|\mathbf{v}) = \prod_j p(h_j|\mathbf{v})$ factorises,
   and derive the sigmoid conditional $p(h_j=1|\mathbf{v})=\sigma(b_j+W_j^\top\mathbf{v})$.

10. **CD gradient.**
    Write out the full gradient $\nabla_W \log p(\mathbf{v})$ for an RBM and
    show how CD-1 approximates the negative phase using one step of Gibbs sampling.

---

### Computational exercises


In [ ]:
# ── Exercise C1: Cosine schedule vs. linear schedule ──────────────────────
def cosine_schedule(T, s=0.008):
    """
    Nichol & Dhariwal (2021) cosine schedule.
    Avoids too-small SNR at the start of the linear schedule.
    """
    steps = T + 1
    t     = torch.linspace(0, T, steps)
    f     = torch.cos(((t / T + s) / (1 + s)) * math.pi / 2).pow(2)
    acp   = f / f[0]
    betas_cos = 1 - acp[1:] / acp[:-1]
    return betas_cos.clamp(min=1e-5, max=0.999)


betas_lin = torch.linspace(BETA_START, BETA_END, T)
betas_cos = cosine_schedule(T)
acp_lin   = torch.cumprod(1 - betas_lin, 0)
acp_cos   = torch.cumprod(1 - betas_cos, 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_range   = range(T)

axes[0].plot(t_range, betas_lin.numpy(), label='Linear', lw=2)
axes[0].plot(t_range, betas_cos.numpy(), label='Cosine', lw=2)
axes[0].set_xlabel('t'); axes[0].set_ylabel(r'$\beta_t$')
axes[0].set_title('Noise schedules $\\beta_t$')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_range, (acp_lin/(1-acp_lin+1e-8)).log().numpy(), label='Linear', lw=2)
axes[1].plot(t_range, (acp_cos/(1-acp_cos+1e-8)).log().numpy(), label='Cosine', lw=2)
axes[1].set_xlabel('t'); axes[1].set_ylabel(r'$\log$ SNR')
axes[1].set_title('Signal-to-Noise Ratio (log scale)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Exercise C1: Compare linear vs cosine noise schedules',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Try training a DDPM with the cosine schedule and compare sample quality.')


In [ ]:
# ── Exercise C2: Effect of latent dimension on VAE quality ────────────────
print('Exercise C2: Train VAEs with different latent dimensions')
print('and compare FID proxy and latent-space structure.')
print()

# Quick sweep: 2, 8, 20, 64 latent dims with 5 epochs each
latent_dims  = [2, 8, 20, 64]
fid_by_dim   = {}

for ldim in latent_dims:
    vae_exp = VAE(latent_dim=ldim).to(device)
    opt_exp = torch.optim.Adam(vae_exp.parameters(), lr=1e-3)
    for _ in range(5):   # 5 epochs for the sweep
        for x, _ in loader_train_01:
            x_hat, mu, lv = vae_exp(x.to(device))
            loss, _, _ = vae_loss(x, x_hat, mu, lv)
            opt_exp.zero_grad(); loss.backward(); opt_exp.step()

    vae_exp.eval()
    with torch.no_grad():
        z_s   = torch.randn(256, ldim, device=device)
        samps = torch.sigmoid(vae_exp.decode(z_s)).cpu()
    mu_s, cov_s = pixel_stats(samps, n_max=256)
    fid_by_dim[ldim] = proxy_fid(mu_real, cov_real, mu_s, cov_s)
    print(f'  Latent dim = {ldim:3d}  |  Proxy-FID = {fid_by_dim[ldim]:.2f}')

fig_c2, ax_c2 = plt.subplots(figsize=(7, 4))
ax_c2.plot(list(fid_by_dim.keys()), list(fid_by_dim.values()),
         'o-', lw=2, ms=8, color='steelblue')
ax_c2.set_xlabel('Latent dimension $d_h$'); ax_c2.set_ylabel('Proxy-FID (lower = better)')
ax_c2.set_title('Exercise C2: VAE quality vs. latent dimension (5 epochs each)')
ax_c2.set_xscale('log'); ax_c2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('\nNote: small latent dim underfits; large dim can cause posterior collapse.')


In [ ]:
# ── Exercise C3: beta-VAE — balancing reconstruction and KL ───────────────
print('Exercise C3: Effect of beta on VAE (beta-VAE disentanglement)')
print()

betas_to_test = [0.5, 1.0, 4.0]
fig, axes = plt.subplots(len(betas_to_test), 8, figsize=(16, 3*len(betas_to_test)))

for row_idx, beta_val in enumerate(betas_to_test):
    vae_b = VAE(latent_dim=20).to(device)
    opt_b = torch.optim.Adam(vae_b.parameters(), lr=1e-3)
    for _ in range(5):
        for x, _ in loader_train_01:
            x_hat, mu, lv = vae_b(x.to(device))
            loss, _, _ = vae_loss(x, x_hat, mu, lv, beta=beta_val)
            opt_b.zero_grad(); loss.backward(); opt_b.step()

    vae_b.eval()
    with torch.no_grad():
        z_b   = torch.randn(8, 20, device=device)
        samps = torch.sigmoid(vae_b.decode(z_b)).cpu()

    for col in range(8):
        axes[row_idx, col].imshow(samps[col].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[row_idx, col].axis('off')
    axes[row_idx, 0].set_ylabel(f'beta={beta_val}', fontsize=12, rotation=0,
                                labelpad=50, va='center')

plt.suptitle(r'Exercise C3: $\beta$-VAE samples (5 epochs) — '
             r'larger $\beta$ regularises more strongly',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Larger beta -> more structured latent space, but blurrier reconstructions.')
print('Smaller beta -> sharper reconstructions, but latent space may collapse.')


---
## Summary

Both VAEs and Diffusion Models are **principled probabilistic generative models**
rooted in the same variational lower bound.
GANs and Energy-Based Models offer complementary trade-offs.
The unifying picture:

$$\underbrace{\text{RBM/EBM}}_{\text{energy, MCMC}}
\xrightarrow{\text{variational}}
\underbrace{\text{VAE}}_{\text{1 level, compact latent}}
\xrightarrow{T\to\infty\text{ levels}}
\underbrace{\text{Diffusion}}_{\text{T levels, full-dim latent}}
\qquad\qquad
\underbrace{\text{GAN}}_{\text{adversarial, no likelihood}}$$

| Design axis | RBM | VAE | DDPM | GAN |
|-------------|-----|-----|------|-----|
| Latent levels | 1–$L$ | 1 | $T$ | 0 (implicit) |
| Encoder learned? | Gibbs | Yes | No (fixed) | No encoder |
| Likelihood available? | $\approx$ | ELBO | ELBO | No |
| Sample quality | Low | Medium | High | High |
| Sampling speed | Slow | Fast | Slow | Fast |
| Training stability | Moderate | High | High | Low–Moderate |

**Latent Diffusion Models** (Stable Diffusion) combine a VAE encoder
with a DDPM operating in the compressed latent space — inheriting the
sample quality of diffusion and the efficiency of the VAE's compact representation.
